# Gemini 3.5 Flash Lite APIで赤ちゃん・ママ言葉変換

このノートブックは、Google AI Studio経由で Gemini 3.5 Flash Lite API を呼び出し、
文章を赤ちゃん・園児語またはママ・やさしい口調に変換します。

**モデルについて:** Issue #15 の当初指定は Gemma 4 31B でしたが、
知能が不足していたため、人間監督の判断で Gemini 3.5 Flash Lite へ変更しました。

**必要なもの:**
- Google AI Studioで取得したAPIキー
- インターネット接続

**特徴:**
- ローカルモデル不要
- GPUなしで実行可能
- リアルタイム変換
- 150文字制限と自動再試行


## セットアップ

### 1. 依存ライブラリをインストール

In [ ]:
# 旧SDK(google-generativeai)は提供終了。新SDK(google-genai)を使います。
!pip install -q -U google-genai

### 2. APIキーを設定

下のセルを実行し、Google AI Studioから取得したAPIキーを入力してください。

APIキーの取得方法:
1. https://aistudio.google.com/app/apikey にアクセス
2. 「APIキーを作成」をクリック
3. 「新しいプロジェクトでAPIキーを作成」または既存プロジェクトを選択
4. 表示されたキーをコピー

In [ ]:
import getpass
from google import genai
from google.genai import types

api_key = getpass.getpass("Google AI Studio APIキーを入力してください: ")
client = genai.Client(api_key=api_key)
print("✓ APIキーが設定されました。")

In [ ]:
# 利用可能なモデルを確認する（Issue #15：識別子は実測で確定させる）
print("generateContentに対応したモデル:")
print("=" * 60)
for m in client.models.list():
    actions = getattr(m, "supported_actions", None) or []
    if "generateContent" in actions:
        print(f"  {m.name}")

## 実装

In [ ]:
from typing import Literal
from google.genai import types

MODEL_NAME = "gemini-3.5-flash-lite"
MAX_OUTPUT_CHARS = 150
USE_FEWSHOT = True          # 手本を会話ターンとして与える。文体の強さはここが効きます
TEMPERATURE = 0.6           # 低すぎると原文に貼り付きます
DEBUG_SHOW_REQUEST = False  # Trueにすると、実際に送る内容を表示します

# ===== 指定モデルが実在するか確認する（Issue #15：識別子は実測で確定させる）=====
available = [
    (m.name or "").replace("models/", "")
    for m in client.models.list()
    if "generateContent" in (getattr(m, "supported_actions", None) or [])
]
if MODEL_NAME not in available:
    print(f"[!] {MODEL_NAME} が一覧にありません。利用できるflash系の候補:")
    for name in available:
        if "flash" in name:
            print(f"    {name}")
    raise RuntimeError(f"MODEL_NAME を上の候補から選び直してください（現在: {MODEL_NAME}）")
print(f"OK モデル {MODEL_NAME} を確認しました。")

# ===== 指示部（Issue #15の本文 ＋ 言い換えの手がかり）=====
# 「言い換えのしかた」は Issue #15 の基準プロンプトへの追加です。
# 基準のままだと語彙も文末も大人のままで、原文に近い出力になったため足しました。
INSTRUCTIONS = {
    "baby": """あなたは文章の言い換え器です。
入力文の意味・事実・感情を保ったまま、「幼児退行した人が話す自然な赤ちゃん・園児語」に言い換えてください。

ルール:
- 入力への返答や助言はしない。入力文そのものを言い換える
- 原文にない情報・感情・解決策を追加しない
- 技術用語、製品名、数値、英数字はなるべくそのまま残す
- ひらがなを少し多めにし、短く幼い言い回しにする
- 「ばぶ」「おぎゃー」「でちゅ」などは多用しない
- かわいさより、元の意味が伝わることを優先する
- 150文字以内
- 絵文字、Markdown、説明、注釈は出力しない
- 変換後の文章だけを出力する

言い換えのしかた:
- 漢字はできるだけひらがなにし、分かち書きぎみにする
- むずかしい言葉を、3〜5歳が使う言葉に置きかえる
  （調査する→しらべる／整理する→おかたづけする／発生する→でちゃう／
    確認する→みてみる／実装する→つくる／指摘→だめだし）
- 文末を「〜なの」「〜のー」「〜ちゃった」「〜だもん」などにする
- 一人称は「ぼく」「わたち」にする
- 敬語やビジネス表現は使わない
- ただし製品名・英数字・数値はひらがなにせず、そのまま残す
  （React.js、index.ts、150、v2 などは変えない）""",

    "mother": """あなたは文章の言い換え器です。
入力文の意味をできるだけ保ったまま、「やさしく包み込むお母さん・ママ口調」に言い換えてください。

ルール:
- 入力への返答はしない。入力文そのものを言い換える
- 原文にない出来事・感情・解決策を追加しない
- 命令、説教、冷たい表現、マサカリ表現をやわらかくする
- 必要な助言が原文にある場合は、内容を消さず任意の提案表現へ変える
- 相手の能力や人格を否定する表現は、責めない表現へ変える
- 技術用語、製品名、数値、英数字はなるべくそのまま残す
- 「よしよし」「えらいね」などは必要な場合だけ使い、多用しない
- 150文字以内
- 絵文字、Markdown、説明、注釈は出力しない
- 変換後の文章だけを出力する

言い換えのしかた:
- 断定を和らげる（「〜だ」「〜しろ」→「〜ね」「〜のね」「〜かな」）
- 命令を、相手に選ばせる問いかけに変える（「調べろ」→「調べてもらえるかな」）
- 責める言い方を、事実を確かめる言い方に変える
  （「なんでこうした」→「どうしてそうしたのか、聞かせてもらえるかな」）
- 語尾に「ね」「かな」「のね」を置いて、話しかける調子にする
- ただし製品名・英数字・数値はそのまま残す""",
}

# ===== 入力側の定型文（Issue #15の指定どおり）=====
ASK = {
    "baby": "次の文章を赤ちゃん・園児語へ言い換えてください。",
    "mother": "次の文章をやさしいお母さん・ママ口調へ言い換えてください。",
}

# ===== 手本 =====
# ここが文体の強さを決めます。弱い手本を置くと弱い出力になります。
# 原文にない情報・感情を足していない例だけを置くこと。
# 3つ目は「製品名は変えない」を教えるための例です。
EXAMPLES = {
    "baby": [
        ("明日までに資料を作らないといけない。",
         "あしたまでに しりょう つくらなきゃ だめなのー。"),
        ("エラーが発生したので、原因を調査してください。",
         "エラー でちゃったのー。どうして でちゃったか しらべて ほしいのー。"),
        ("React.js のバージョンで詰まっている。",
         "React.js の ばーじょんで つまっちゃったのー。"),
    ],
    "mother": [
        ("なんでこんな設計にしたの。ありえない。",
         "どうしてこの設計にしたのか、聞かせてもらえるかな。"),
        ("エラーが発生したので、原因を調査してください。",
         "エラーが出てしまったのね。原因を調べてもらえるかな。"),
        ("React.js のバージョンで詰まっている。",
         "React.js のバージョンのところで、詰まってしまっているのね。"),
    ],
}


def build_contents(mode, text, retry=False, fewshot=None):
    """会話のターンとして組み立てる。

    ルール本文は system_instruction 側に置くので、ここには含めない。
    Issue #15 が指定した「system部」と「入力側」の分離をそのまま実装している。
    """
    if fewshot is None:
        fewshot = USE_FEWSHOT

    turns = []
    if fewshot:
        for source_text, target_text in EXAMPLES[mode]:
            turns.append({"role": "user", "parts": [{"text": f"{ASK[mode]}\n\n{source_text}"}]})
            turns.append({"role": "model", "parts": [{"text": target_text}]})

    ask = f"{ASK[mode]}\n\n{text}"
    if retry:
        ask += "\n\n（前回は150文字を超えました。意味を保って、必ず150文字以内へ短くしてください。）"
    turns.append({"role": "user", "parts": [{"text": ask}]})
    return turns


def clean_output(raw):
    text = raw.strip()
    for prefix in ("出力:", "出力："):
        if text.startswith(prefix):
            text = text[len(prefix):]
    if text.strip().startswith("```"):
        text = "\n".join(line for line in text.strip().split("\n") if not line.startswith("```"))
    return text.strip()


def _call_api(mode, contents, thinking_off=True):
    """1回だけAPIを呼ぶ。thinking設定が非対応なら1度だけ外して呼び直す。

    Gemini 3系は既定で思考にトークンを使う。思考だけで上限に達すると
    本文が空で返るため、変換タスクでは思考を切る。
    """
    kwargs = {
        "temperature": TEMPERATURE,
        "max_output_tokens": 512,
        "system_instruction": INSTRUCTIONS[mode],
    }
    if thinking_off:
        kwargs["thinking_config"] = types.ThinkingConfig(thinking_budget=0)

    try:
        return client.models.generate_content(
            model=MODEL_NAME,
            contents=contents,
            config=types.GenerateContentConfig(**kwargs),
        )
    except Exception as exc:
        message = str(exc)
        if thinking_off and ("400" in message or "INVALID_ARGUMENT" in message):
            return _call_api(mode, contents, thinking_off=False)
        raise


def _extract_text(response):
    """response.text が空でも、candidatesから拾えるだけ拾う。"""
    direct = getattr(response, "text", None)
    if direct and direct.strip():
        return direct
    for cand in (getattr(response, "candidates", None) or []):
        parts = getattr(getattr(cand, "content", None), "parts", None) or []
        joined = "".join(getattr(p, "text", "") or "" for p in parts)
        if joined.strip():
            return joined
    return ""


def _describe(response):
    """空応答のとき、原因を人が読める形にする。"""
    bits = []
    for cand in (getattr(response, "candidates", None) or []):
        bits.append(f"finish_reason={getattr(cand, 'finish_reason', '不明')}")
        bits.append(f"safety={getattr(cand, 'safety_ratings', None)}")
    if not bits:
        bits.append("candidatesが空")
    bits.append(f"usage={getattr(response, 'usage_metadata', None)}")
    bits.append(f"prompt_feedback={getattr(response, 'prompt_feedback', None)}")
    return " / ".join(str(b) for b in bits)


def transform_text(mode: Literal["baby", "mother"], text: str,
                   retry: bool = False, fewshot=None) -> str:
    """文章を指定のスタイルへ言い換える。

    Args:
        mode: 変換スタイル ("baby" または "mother")
        text: 入力文章
        retry: 150文字超過時の再試行フラグ
        fewshot: 手本を使うか。Noneなら USE_FEWSHOT に従う

    Returns:
        変換後の文章
    """
    if mode not in ("baby", "mother"):
        raise ValueError(f"modeは'baby'または'mother'です。指定: {mode}")
    if not text or not text.strip():
        raise ValueError("空文字列は受け付けません。")
    if len(text) > 500:
        raise ValueError(f"入力は500文字以内です。現在: {len(text)}文字")

    contents = build_contents(mode, text, retry=retry, fewshot=fewshot)

    if DEBUG_SHOW_REQUEST:
        print("----- system_instruction -----")
        print(INSTRUCTIONS[mode])
        print("----- 送信する会話 -----")
        for turn in contents:
            print(f"[{turn['role']}] {turn['parts'][0]['text']}")
        print("----- ここまで -----")

    try:
        response = _call_api(mode, contents)
    except Exception as exc:
        message = str(exc)
        if "429" in message or "RESOURCE_EXHAUSTED" in message:
            raise RuntimeError("APIレート制限。しばらく待ってから再試行してください。") from exc
        if "401" in message or "403" in message or "PERMISSION_DENIED" in message:
            raise RuntimeError("認証エラー。APIキーを確認してください。") from exc
        if "404" in message or "NOT_FOUND" in message:
            raise RuntimeError(f"モデル {MODEL_NAME} が見つかりません。") from exc
        raise RuntimeError(f"API呼び出しエラー: {exc}") from exc

    raw = _extract_text(response)
    if not raw.strip():
        raise RuntimeError("APIが空の応答を返しました。" + _describe(response))
    return clean_output(raw)

print(f"✓ 実装完了（モデル: {MODEL_NAME}）。以下のセルで変換を実行してください。")

## 使い方

下のセルで入力文とモード（赤ちゃん/ママ）を指定して実行してください。

In [ ]:
# 赤ちゃん・園児語への変換例
input_text = "今日はチームで仕様書をレビューし、未決事項を整理しました。"

print(f"入力: {input_text}")
print(f"入力文字数: {len(input_text)}")
print()

try:
    output = transform_text("baby", input_text)
    print(f"赤ちゃん・園児語: {output}")
    print(f"出力文字数: {len(output)}")
except Exception as e:
    print(f"エラー: {e}")

In [ ]:
# ママ・やさしい口調への変換例
input_text = "今日はチームで仕様書をレビューし、未決事項を整理しました。"

print(f"入力: {input_text}")
print(f"入力文字数: {len(input_text)}")
print()

try:
    output = transform_text("mother", input_text)
    print(f"ママ・やさしい口調: {output}")
    print(f"出力文字数: {len(output)}")
except Exception as e:
    print(f"エラー: {e}")

## 対話的に使う

このセルを編集して、好きな文章とモードを試してください。

In [ ]:
# ここを編集して試してください
my_text = "これはテストです。自由に編集して試してください。"
my_mode = "baby"  # "baby" または "mother"

print(f"入力: {my_text}")
print(f"モード: {my_mode}")
print(f"入力文字数: {len(my_text)}")
print("-" * 50)

try:
    output = transform_text(my_mode, my_text)
    print(f"変換結果: {output}")
    print(f"出力文字数: {len(output)}")

    if len(output) > MAX_OUTPUT_CHARS:
        print(f"\n⚠️  150文字を超えたため、再試行します...")
        output = transform_text(my_mode, my_text, retry=True)
        print(f"再試行結果: {output}")
        print(f"出力文字数: {len(output)}")
except Exception as e:
    print(f"❌ エラー: {e}")

## 複数の文章を一括処理

複数の文章をまとめて変換したい場合はこのセルを使ってください。

In [ ]:
# 変換対象の文章リスト
texts = [
    "今日はチームで仕様書をレビューし、未決事項を整理しました。",
    "テストがパスしました。",
    "エラーが発生したので、原因を調査してください。",
]

mode = "baby"  # "baby" または "mother"

print(f"モード: {mode}")
print("=" * 60)

results = []
for i, text in enumerate(texts, 1):
    print(f"\n[{i}] 入力: {text}")
    try:
        output = transform_text(mode, text)
        print(f"    出力: {output}")
        results.append({
            "input": text,
            "output": output,
            "mode": mode,
            "success": True,
        })
    except Exception as e:
        print(f"    ❌ エラー: {e}")
        results.append({
            "input": text,
            "error": str(e),
            "mode": mode,
            "success": False,
        })

print("\n" + "=" * 60)
print(f"処理完了: {sum(1 for r in results if r['success'])}/{len(results)} 成功")

## 結果をJSONで保存

変換結果をJSONファイルとして保存できます。

In [ ]:
# 結果をJSON形式で表示
if results:
    print(json.dumps(results, ensure_ascii=False, indent=2))
else:
    print("結果がありません。上のセルで処理を実行してください。")

## 注意事項

- **APIキー:** このノートブックの実行ログには、処理した文章が記録されます。個人情報を含まない内容で試してください。
- **レート制限:** Google AI Studioの無料枠にはレート制限があります。大量の処理が必要な場合は間隔を開けてください。
- **出力:** 150文字を超える場合、自動的に1回だけ再試行します。それでも超過した場合はエラーになります。
- **品質:** APIからの出力はモデルの確率的な生成です。同じ入力でも結果が異なることがあります。

## サポート

エラーが出た場合:
1. APIキーが正しく設定されているか確認
2. インターネット接続を確認
3. 入力文の長さを確認（最大500文字）
4. レート制限に達していないか確認（429エラー）

## SDKについて

旧SDK `google-generativeai` は提供終了しました。このノートブックは新SDK `google-genai` を使います。
既存のColabセッションに旧SDKが残っている場合は、ランタイムを再起動してから最初のセルを実行してください。
